# Week 3, day 2 — Worksheet 03 SOLUTIONS: unstack   (L04)

Executed in the lab image (pandas 3.0.5) against the real
`data/orders_long.csv`. Every quoted number is what it actually printed.

Questions 4 and 5 are the ones to re-read. Unstacking invented cells that no
order ever filled, and changed the dtype of a column you did not touch.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 03 — unstack. Run this once.
import pandas as pd

orders = pd.read_csv("data/orders_long.csv")

# Every combination is present in the full data...
by_ry = orders.groupby(["Region", "Year"])["Sales"].sum().round(2)

# ...but Technology has gaps: no Nunavut orders in 2009 or 2012.
tech = orders[orders["Category"] == "Technology"]
by_tech = tech.groupby(["Region", "Year"])["Sales"].sum().round(2)

print("all orders:", len(by_ry), "combinations")
print("technology:", len(by_tech), "combinations")

PART A — the operation

### Question 1

A `(8, 4)` frame: regions down, **years across**. -> `unstack()` moved the **innermost** level.

You did not say which level to move and it took the last one. That is the
documented default and it is worth stating explicitly anyway, because the
level order came from `groupby(["Region", "Year"])` — change the order of
those two names and the same `unstack()` call produces the transpose.

Note the columns are **integers** (`2009`), not strings, because `Year` is
`int64` in the file. `wide["2009"]` raises; `wide[2009]` works.

In [ ]:
wide = by_ry.unstack()
print(wide.to_string())
print()
print("shape:", wide.shape)
print("columns:", list(wide.columns))
print("index name:", wide.index.name)

### Question 2

`unstack()` and `unstack("Year")` are identical. -> `unstack("Region")` gives the transpose: years down, regions across.

Naming the level costs four words and removes the dependence on level
order entirely. Worth doing in anything you will read again.

The transposed version is the same data and answers different questions
comfortably — 'how did 2012 compare across regions' reads down a column
instead of across a row.

In [ ]:
a = by_ry.unstack()
b = by_ry.unstack("Year")
print("unstack() == unstack('Year'):", a.equals(b))
print()
print("unstack('Region'):")
print(by_ry.unstack("Region").to_string())

### Question 3

`by_ry.unstack().stack()` -> `32` rows, `.equals()` the original: **`True`**.

A clean round trip, because every combination is present so the grid has
no holes to invent or drop.

Hold this result. Q6 does the same thing to data with gaps and the answer
changes.

In [ ]:
rt = by_ry.unstack().stack()
print(rt.head(6).to_string())
print()
print("round trip equals the original:", rt.equals(by_ry))
print("same length:", len(rt), "vs", len(by_ry))

PART B — the cells nobody filled

### Question 4

`30` rows in -> `32` cells out, **`2` of them `NaN`**.

Two cells appeared that no order ever filled: Nunavut in 2009 and 2012.

Nothing was added and nothing was lost — the reshape simply made the grid
rectangular, and a rectangle has room for combinations your data does not
contain. Long format had no row for them and therefore no way to show them.

This is unstacking used as a **check**. If you want to know which expected
combinations are absent, making the shape rectangular is the fastest way to
see them.

In [ ]:
wide = by_tech.unstack()
print(wide.to_string())
print()
print("rows in:", len(by_tech))
print("cells out:", wide.size)
print("NaN cells:", int(wide.isna().sum().sum()))

### Question 5

`by_tech` is `float64` before and after. -> the complete data is `float64` too.

No dtype change here, because `Sales` was already a float — the sum of
floats is a float, and `NaN` fits in it without widening.

That is the exception rather than the rule. Had you unstacked a **count** or
a **quantity** — genuine `int64` columns — the two `NaN` cells would have
forced the whole frame to `float64`, exactly as in the week 3, day 1 class's
reindexing sheet. Try it with
`tech.groupby(["Region","Year"]).size().unstack()` and watch the integers
become floats.

In [ ]:
print("before unstack:", by_tech.dtype)
print("after unstack: ", by_tech.unstack().dtypes.unique())
print()
print("and for the complete data:")
print("before:", by_ry.dtype)
print("after: ", by_ry.unstack().dtypes.unique())

### Question 6

`fill_value=0` -> Nunavut reads `0.00` in 2009 and 2012, dtype still `float64`.

Here zero is defensible, and it is worth being explicit about *why*:
Nunavut genuinely sold no Technology in those years, so 'zero sales' is a
true statement about the world.

It would not be defensible for a temperature, a price, a rate or a score,
where an absent reading means 'unknown' and zero is a specific, wrong claim.
The deck's warning is exactly right — `fill_value` is a statement, not a
formatting option.

In [ ]:
filled = by_tech.unstack(fill_value=0)
print(filled.to_string())
print()
print("dtypes:", filled.dtypes.unique())

# Here 0 is defensible: Nunavut sold no Technology in 2009, so zero sales
# is literally true. It would NOT be defensible for a temperature, a price,
# or a rate -- where a missing reading means "unknown", not "none".

### Question 7

Column **totals are identical** either way. But the 2009 **mean** is `37207.21` with `NaN` and `32556.31` with zeros — over `7` rows versus `8`.

This is the question. The choice of `fill_value` changed one summary and
not the other, and nothing in the code says so.

`sum()` skips `NaN`, and adding zero changes nothing, so the totals agree —
correctly, both times.

`mean()` also skips `NaN`, but skipping changes the **denominator**. With
`NaN` the mean is over the 7 regions that sold Technology; with zeros it is
over all 8. Both are real numbers and they answer different questions:
'average among regions that sell it' versus 'average across the whole
territory'. A 14% gap between them, decided by an argument passed to a
reshaping function three steps earlier.

When a reader asks 'what is the average', the useful reply is 'average over
what?'.

In [ ]:
plain = by_tech.unstack()
filled = by_tech.unstack(fill_value=0)
print("totals with NaN: ", plain.sum().round(2).to_dict())
print("totals with 0:   ", filled.sum().round(2).to_dict())
print()
# NOTE the column labels are INTEGERS -- Year is int64 in the file, so it
# is plain[2009], not plain["2009"].
print("columns:", list(plain.columns))
print("mean with NaN: ", round(plain[2009].mean(), 2))
print("mean with 0:   ", round(filled[2009].mean(), 2))
print("rows counted:  ", plain[2009].count(), "vs", filled[2009].count())

PART C — on a three-level index

### Question 8

`three.unstack("Category")` -> `(32, 3)`, index levels `['Region', 'Year']`, **6** `NaN` cells.

Only the named level moved; the other two stayed as rows. That is how you
get a table with a compound row label and one clean set of columns — often
the most readable form for a report.

The 6 `NaN`s are the missing combinations from worksheet 02 Q9, now
visible as cells rather than as absent rows.

In [ ]:
three = orders.groupby(["Region", "Year", "Category"])["Sales"].sum().round(2)
out = three.unstack("Category")
print("shape:", out.shape)
print("remaining index levels:", out.index.names)
print()
print(out.head(6).to_string())
print()
print("NaN cells:", int(out.isna().sum().sum()))

### Question 9

`unstack(["Year", "Category"])` -> `(8, 12)` with a **2-level column index**, e.g. `(2009, 'Furniture')`. -> 96 cells, `90` filled, `6` `NaN`.

Both levels moved, so the columns became a MultiIndex of their own and the
rows collapsed to one per region.

94% filled, so the grid is not sparse — but it is 12 columns wide for 8
rows of data, and each column name is a pair. Selecting from it needs
tuples: `out[(2009, "Furniture")]`.

That is the trade. Unstacking more levels makes a table you can read in one
glance and awkward to compute with. Reach for it at the end of an analysis,
not in the middle.

In [ ]:
three = orders.groupby(["Region", "Year", "Category"])["Sales"].sum().round(2)
out = three.unstack(["Year", "Category"])
print("shape:", out.shape)
print("column levels:", out.columns.nlevels)
print("first 4 columns:", list(out.columns[:4]))
print()
print("cells:", out.size, "| filled:", int(out.notna().sum().sum()),
      "| NaN:", int(out.isna().sum().sum()))

### Question 10

`by_ry.unstack("Category")` -> **raises** `KeyError: 'Level Category not found'`.

`by_ry` has levels `['Region', 'Year']` only — `Category` was never
grouped on, so there is no level to move.

The message names the level, which is more helpful than most. And because
naming the level is what makes the error possible, this is a small argument
for always naming it: `unstack()` with no argument cannot fail this way, it
just silently moves whichever level happens to be innermost.

In [ ]:
print("levels available:", by_ry.index.names)
print(by_ry.unstack("Category"))